# RepresentationLayer Demonstration

All initialization distributions available in `RepresentationLayer`, with visualizations and basic statistics for each.

## Import Required Libraries

Import necessary libraries for neural network operations, visualization, and statistical analysis.

In [ ]:
import sys
import os
import torch
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from mpl_toolkits.mplot3d import Axes3D
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# Add src path for imports
sys.path.append('../src/models')
from representation_layer import RepresentationLayer

# reload the module to get latest changes
import importlib
importlib.reload(sys.modules['representation_layer'])


# Set style
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

print("Libraries imported successfully!")
print(f"PyTorch version: {torch.__version__}")
print(f"Device: {'CUDA' if torch.cuda.is_available() else 'CPU'}")

## Configuration and Helper Functions

Define configuration parameters and utility functions for visualization and statistical analysis.

In [ ]:
# Configuration
N_SAMPLES = 1000
DIM_2D = 2
DIM_3D = 3
FIGSIZE = (15, 10)
SEED = 42

# Set random seeds for reproducibility
torch.manual_seed(SEED)
np.random.seed(SEED)

def plot_2d_scatter(data, title, ax=None):
    """Plot 2D scatter plot with statistics."""
    if ax is None:
        fig, ax = plt.subplots(figsize=(8, 6))
    
    data_np = data.detach().cpu().numpy()
    ax.scatter(data_np[:, 0], data_np[:, 1], alpha=0.6, s=20)
    ax.set_title(f"{title}\nMean: ({data_np[:, 0].mean():.3f}, {data_np[:, 1].mean():.3f})\nStd: ({data_np[:, 0].std():.3f}, {data_np[:, 1].std():.3f})")
    ax.set_xlabel('Dimension 1')
    ax.set_ylabel('Dimension 2')
    ax.grid(True, alpha=0.3)
    return ax

def plot_3d_scatter(data, title):
    """Plot 3D scatter plot."""
    fig = plt.figure(figsize=(10, 8))
    ax = fig.add_subplot(111, projection='3d')
    
    data_np = data.detach().cpu().numpy()
    ax.scatter(data_np[:, 0], data_np[:, 1], data_np[:, 2], alpha=0.6, s=20)
    ax.set_title(title)
    ax.set_xlabel('Dimension 1')
    ax.set_ylabel('Dimension 2')
    ax.set_zlabel('Dimension 3')
    return fig, ax

def plot_histogram(data, title, ax=None, xlim=None, bin_width=0.25):
    """Plot histogram with statistical information and consistent bin width."""
    if ax is None:
        fig, ax = plt.subplots(figsize=(8, 6))
    
    data_flat = data.detach().cpu().numpy().flatten()
    
    # Calculate bins based on fixed bin width for consistent resolution
    if xlim is not None:
        x_range = xlim[1] - xlim[0]
        n_bins = int(x_range / bin_width)
        # Create bin edges from xlim[0] to xlim[1] with fixed width
        bins = np.linspace(xlim[0], xlim[1], n_bins + 1)
    else:
        bins = 50
    
    ax.hist(data_flat, bins=bins, alpha=0.7, density=True, edgecolor='black', linewidth=0.5)
    ax.set_title(f"{title}\nMean: {data_flat.mean():.3f}, Std: {data_flat.std():.3f}\nSkew: {stats.skew(data_flat):.3f}, Kurt: {stats.kurtosis(data_flat):.3f}")
    ax.set_xlabel('Value')
    ax.set_ylabel('Density')
    ax.grid(True, alpha=0.3)
    
    # Set x-axis limits if provided
    if xlim is not None:
        ax.set_xlim(xlim)
    
    return ax


def print_statistics(data, dist_name):
    """Print comprehensive statistics for the data."""
    data_flat = data.detach().cpu().numpy().flatten()
    print(f"\n{dist_name} Distribution Statistics:")
    print(f"Shape: {data.shape}")
    print(f"Mean: {data_flat.mean():.4f}")
    print(f"Std: {data_flat.std():.4f}")
    print(f"Min: {data_flat.min():.4f}")
    print(f"Max: {data_flat.max():.4f}")
    print(f"Skewness: {stats.skew(data_flat):.4f}")
    print(f"Kurtosis: {stats.kurtosis(data_flat):.4f}")
    print("-" * 40)

print("Helper functions defined successfully!")

## Normal Distribution

$$\mathbf{X} \sim \mathcal{N}_k(\boldsymbol{\mu}, \boldsymbol{\Sigma}), \qquad f(\mathbf{x}) = \frac{1}{(2\pi)^{k/2}|\boldsymbol{\Sigma}|^{1/2}} \exp\left(-\tfrac{1}{2}(\mathbf{x}-\boldsymbol{\mu})^T\boldsymbol{\Sigma}^{-1}(\mathbf{x}-\boldsymbol{\mu})\right)$$

Scalar `mean`/`cov` broadcast to $\mu\mathbf{1}$ / $\mathrm{cov}\cdot\mathbf{I}_k$ (spherical). Standard init distribution for continuous latents.

In [ ]:
print("=== NORMAL DISTRIBUTION DEMONSTRATION ===")

# Use CPU device for consistency
device = torch.device('cpu')

# Create normal distributions with different parameters
normal_configs = [
    {'mean': torch.zeros(DIM_2D, device=device), 'cov': torch.eye(DIM_2D, device=device), 'title': 'Standard Normal (μ=0, σ=1)'},
    {'mean': torch.zeros(DIM_2D, device=device), 'cov': 0.5**2 * torch.eye(DIM_2D, device=device), 'title': 'Narrow Normal (μ=0, σ=0.5)'},
    {'mean': torch.ones(DIM_2D, device=device), 'cov': torch.eye(DIM_2D, device=device), 'title': 'Shifted Normal (μ=1, σ=1)'},
    {'mean': torch.zeros(DIM_2D, device=device), 'cov': 2.0**2 * torch.eye(DIM_2D, device=device), 'title': 'Wide Normal (μ=0, σ=2)'}
]

# 2D visualizations
fig, axes = plt.subplots(2, 4, figsize=(20, 10))
fig.suptitle('Normal Distribution Initializations', fontsize=16)

for i, config in enumerate(normal_configs):
    # Create representation layer
    layer = RepresentationLayer(
        dim=DIM_2D, 
        n_samples=N_SAMPLES, 
        dist='normal',
        dist_params={'mean': config['mean'], 'cov': config['cov']},
        device=device
    )
    
    # Print statistics
    print_statistics(layer.z, config['title'])
    
    # Scatter plot with consistent limits
    plot_2d_scatter(layer.z, config['title'], axes[0, i])
    axes[0, i].set_xlim(-4, 4)
    axes[0, i].set_ylim(-4, 4)
    
    # Histogram with consistent limits and bin width
    plot_histogram(layer.z, config['title'], axes[1, i], xlim=(-4, 4), bin_width=0.25)

plt.tight_layout()
plt.show()

# 3D visualization for standard normal
layer_3d = RepresentationLayer(dim=DIM_3D, n_samples=N_SAMPLES, dist='normal', device=device)
fig, ax = plot_3d_scatter(layer_3d.z, '3D Standard Normal Distribution')
ax.set_xlim(-4, 4)
ax.set_ylim(-4, 4)
ax.set_zlim(-4, 4)
plt.show()

## Uniform Distribution

$$\mathbf{X} \sim \mathrm{Unif}\big([a_1,b_1]\times\cdots\times[a_k,b_k]\big), \qquad \mathbb{E}[X_i]=\tfrac{a_i+b_i}{2}, \ \mathrm{Var}[X_i]=\tfrac{(b_i-a_i)^2}{12}$$

Scalar `low`/`high` apply the same bound to every dimension.

In [ ]:
print("=== UNIFORM DISTRIBUTION DEMONSTRATION ===")

# Create uniform distributions with different bounds
uniform_configs = [
    {'low': -1.0, 'high': 1.0, 'title': 'Standard Uniform [-1, 1]'},
    {'low': -0.5, 'high': 0.5, 'title': 'Narrow Uniform [-0.5, 0.5]'},
    {'low': 0.0, 'high': 2.0, 'title': 'Positive Uniform [0, 2]'},
    {'low': -2.0, 'high': 2.0, 'title': 'Wide Uniform [-2, 2]'}
]

# 2D visualizations
fig, axes = plt.subplots(2, 4, figsize=(20, 10))
fig.suptitle('Uniform Distribution Initializations', fontsize=16)

for i, config in enumerate(uniform_configs):
    # Create representation layer
    layer = RepresentationLayer(
        dim=DIM_2D, 
        n_samples=N_SAMPLES, 
        dist='uniform',
        dist_params={'low': config['low'], 'high': config['high']}
    )
    
    # Print statistics
    print_statistics(layer.z, config['title'])
    
    # Scatter plot with consistent limits
    plot_2d_scatter(layer.z, config['title'], axes[0, i])
    axes[0, i].set_xlim(-2.5, 2.5)
    axes[0, i].set_ylim(-2.5, 2.5)
    
    # Histogram with consistent limits and bin width
    plot_histogram(layer.z, config['title'], axes[1, i], xlim=(-2.5, 2.5), bin_width=0.25)

plt.tight_layout()
plt.show()

# 3D visualization for standard uniform
layer_3d = RepresentationLayer(dim=DIM_3D, n_samples=N_SAMPLES, dist='uniform')
fig, ax = plot_3d_scatter(layer_3d.z, '3D Standard Uniform Distribution')
ax.set_xlim(-2.5, 2.5)
ax.set_ylim(-2.5, 2.5)
ax.set_zlim(-2.5, 2.5)
plt.show()

## Uniform Ball Distribution

Points uniform over the hyperball $\{\mathbf{y}: \|\mathbf{y}\|\le r\}$: $\mathbb{E}[\mathbf{X}]=\mathbf{0}$, $\mathrm{Var}[X_i]=r^2/(k+2)$. Generated as $\mathbf{X} = r\,U^{1/k}\,\mathbf{Z}/\|\mathbf{Z}\|$ for $\mathbf{Z}\sim\mathcal{N}(\mathbf{0},\mathbf{I}_k)$, $U\sim\mathrm{Unif}(0,1)$. Scalar `radius` sets $r$.

This is the DGD's default representation init (`config.yaml`'s `uniform_ball`, radius 0.1).

In [ ]:
print("=== UNIFORM ball DISTRIBUTION DEMONSTRATION ===")

# Create uniform ball distributions with different radii
ball_configs = [
    {'radius': 1.0, 'title': 'Unit ball (r=1)'},
    {'radius': 0.5, 'title': 'Small ball (r=0.5)'},
    {'radius': 2.0, 'title': 'Large ball (r=2)'},
    {'radius': 1.5, 'title': 'Medium ball (r=1.5)'}
]

# 2D visualizations
fig, axes = plt.subplots(2, 4, figsize=(20, 10))
fig.suptitle('Uniform ball Distribution Initializations', fontsize=16)

for i, config in enumerate(ball_configs):
    # Create representation layer
    layer = RepresentationLayer(
        dim=DIM_2D, 
        n_samples=N_SAMPLES, 
        dist='uniform_ball',
        dist_params={'radius': config['radius']}
    )
    
    # Print statistics
    print_statistics(layer.z, config['title'])
    
    # Scatter plot with circle overlay and consistent limits
    ax = plot_2d_scatter(layer.z, config['title'], axes[0, i])
    circle = plt.Circle((0, 0), config['radius'], fill=False, color='red', linestyle='--', linewidth=2)
    ax.add_patch(circle)
    ax.set_xlim(-2.5, 2.5)
    ax.set_ylim(-2.5, 2.5)
    ax.set_aspect('equal')
    
    # Histogram with consistent limits and bin width
    plot_histogram(layer.z, config['title'], axes[1, i], xlim=(-2.5, 2.5), bin_width=0.25)

plt.tight_layout()
plt.show()

# 3D visualization for unit ball
layer_3d = RepresentationLayer(dim=DIM_3D, n_samples=N_SAMPLES, dist='uniform_ball')
fig, ax = plot_3d_scatter(layer_3d.z, '3D Uniform ball Distribution')
ax.set_xlim(-2.5, 2.5)
ax.set_ylim(-2.5, 2.5)
ax.set_zlim(-2.5, 2.5)
plt.show()

# Verify points are within the ball
distances = torch.norm(layer_3d.z, dim=1)
print(f"\nDistance verification for 3D unit ball:")
print(f"Max distance: {distances.max().detach():.4f} (should be ≤ 1.0)")
print(f"Mean distance: {distances.mean().detach():.4f}")
print(f"Points within unit ball: {(distances <= 1.0).sum().item()}/{len(distances)}")

## Uniform Sphere Distribution

Points uniform on the hypersphere surface $\{\mathbf{y}:\|\mathbf{y}\|=r\}$: $\mathrm{Var}[X_i]=r^2/k$. Generated as $\mathbf{X}=r\,\mathbf{Z}/\|\mathbf{Z}\|$ for $\mathbf{Z}\sim\mathcal{N}(\mathbf{0},\mathbf{I}_k)$ (rotational invariance of the normal makes this uniform on the sphere). Scalar `radius` sets $r$.

Unlike Uniform Ball, mass sits only on the surface, not the interior.

In [ ]:
print("=== UNIFORM SPHERE DISTRIBUTION DEMONSTRATION ===")

# Create uniform sphere distributions with different radii
sphere_configs = [
    {'radius': 1.0, 'title': 'Unit sphere (r=1)'},
    {'radius': 0.5, 'title': 'Small sphere (r=0.5)'},
    {'radius': 2.0, 'title': 'Large sphere (r=2)'},
    {'radius': 1.5, 'title': 'Medium sphere (r=1.5)'}
]

# 2D visualizations
fig, axes = plt.subplots(2, 4, figsize=(20, 10))
fig.suptitle('Uniform Sphere Distribution Initializations', fontsize=16)

for i, config in enumerate(sphere_configs):
    # Create representation layer
    layer = RepresentationLayer(
        dim=DIM_2D, 
        n_samples=N_SAMPLES, 
        dist='uniform_sphere',
        dist_params={'radius': config['radius']}
    )
    
    # Print statistics
    print_statistics(layer.z, config['title'])
    
    # Scatter plot with circle overlay and consistent limits
    ax = plot_2d_scatter(layer.z, config['title'], axes[0, i])
    circle = plt.Circle((0, 0), config['radius'], fill=False, color='red', linestyle='--', linewidth=2)
    ax.add_patch(circle)
    ax.set_xlim(-2.5, 2.5)
    ax.set_ylim(-2.5, 2.5)
    ax.set_aspect('equal')
    
    # Histogram with consistent limits and bin width
    plot_histogram(layer.z, config['title'], axes[1, i], xlim=(-2.5, 2.5), bin_width=0.25)

plt.tight_layout()
plt.show()

# 3D visualization for unit sphere
layer_3d = RepresentationLayer(dim=DIM_3D, n_samples=N_SAMPLES, dist='uniform_sphere')
fig, ax = plot_3d_scatter(layer_3d.z, '3D Uniform Sphere Distribution')
ax.set_xlim(-2.5, 2.5)
ax.set_ylim(-2.5, 2.5)
ax.set_zlim(-2.5, 2.5)
plt.show()

# Verify all points are exactly on the sphere surface
distances = torch.norm(layer_3d.z, dim=1)
print(f"\nDistance verification for 3D unit sphere:")
print(f"Max distance: {distances.max().detach():.6f} (should be ≈ 1.0)")
print(f"Min distance: {distances.min().detach():.6f} (should be ≈ 1.0)")
print(f"Mean distance: {distances.mean().detach():.6f} (should be ≈ 1.0)")
print(f"Std distance: {distances.std().detach():.6f} (should be ≈ 0.0)")
print(f"Distance range: [{distances.min().detach():.6f}, {distances.max().detach():.6f}]")

## Laplace Distribution

Multivariate double-exponential, heavier-tailed than normal: $\mathbb{E}[\mathbf{X}]=\boldsymbol{\mu}$, $\mathrm{Cov}[\mathbf{X}]=\boldsymbol{\Sigma}$, sharply peaked at $\boldsymbol\mu$ with exponential tail decay. Scalar `loc`/`scale_matrix` broadcast to $\mu\mathbf{1}$ / $\mathrm{scale}\cdot\mathbf{I}_k$.

In [ ]:
print("=== LAPLACE DISTRIBUTION DEMONSTRATION ===")

# Use CPU device for now to avoid device conflicts  
device = torch.device('cpu')

# Create Laplace distributions with different parameters
laplace_configs = [
    {'loc': torch.zeros(DIM_2D, device=device), 'scale_matrix': torch.eye(DIM_2D, device=device), 'title': 'Standard Laplace (μ=0, σ=1)'},
    {'loc': torch.zeros(DIM_2D, device=device), 'scale_matrix': 0.5**2 * torch.eye(DIM_2D, device=device), 'title': 'Narrow Laplace (μ=0, σ=0.5)'},
    {'loc': torch.ones(DIM_2D, device=device), 'scale_matrix': torch.eye(DIM_2D, device=device), 'title': 'Shifted Laplace (μ=1, σ=1)'},
    {'loc': torch.zeros(DIM_2D, device=device), 'scale_matrix': 2.0**2 * torch.eye(DIM_2D, device=device), 'title': 'Wide Laplace (μ=0, σ=2)'}
]

# 2D visualizations
fig, axes = plt.subplots(2, 4, figsize=(20, 10))
fig.suptitle('Laplace Distribution Initializations', fontsize=16)

for i, config in enumerate(laplace_configs):
    # Create representation layer on CPU device
    layer = RepresentationLayer(
        dim=DIM_2D, 
        n_samples=N_SAMPLES, 
        dist='laplace',
        dist_params={'loc': config['loc'], 'scale_matrix': config['scale_matrix']},
        device=device
    )
    
    # Print statistics
    print_statistics(layer.z, config['title'])
    
    # Scatter plot with consistent limits
    plot_2d_scatter(layer.z, config['title'], axes[0, i])
    axes[0, i].set_xlim(-4, 4)
    axes[0, i].set_ylim(-4, 4)
    
    # Histogram with consistent limits and bin width
    plot_histogram(layer.z, config['title'], axes[1, i], xlim=(-6, 6), bin_width=0.5)

plt.tight_layout()
plt.show()

# 3D visualization for standard Laplace
layer_3d = RepresentationLayer(dim=DIM_3D, n_samples=N_SAMPLES, dist='laplace', device=device)
fig, ax = plot_3d_scatter(layer_3d.z, '3D Standard Laplace Distribution')
ax.set_xlim(-4, 4)
ax.set_ylim(-4, 4)
ax.set_zlim(-4, 4)
plt.show()

## Student's t-Distribution

$$f(\mathbf{x}) \propto \left[1+\tfrac{1}{\nu}(\mathbf{x}-\boldsymbol\mu)^T\boldsymbol\Sigma^{-1}(\mathbf{x}-\boldsymbol\mu)\right]^{-(\nu+k)/2}$$

Heavier tails than normal, converging to $\mathcal{N}(\boldsymbol\mu,\boldsymbol\Sigma)$ as $\nu\to\infty$ ($\nu=1$ gives multivariate Cauchy). Constructed as $\mathbf{X}=\boldsymbol\mu+\mathbf{Y}/\sqrt{U/\nu}$, $\mathbf{Y}\sim\mathcal N(\mathbf 0,\boldsymbol\Sigma)$, $U\sim\chi^2_\nu$. Scalar `scale_matrix` broadcasts to $\mathrm{scale}\cdot\mathbf I_k$.

In [ ]:
print("=== STUDENT'S T-DISTRIBUTION DEMONSTRATION ===")

# Use CPU device for consistency
device = torch.device('cpu')

# Create Student's t distributions with different parameters
t_configs = [
    {'df': 1.0, 'scale_matrix': torch.eye(DIM_2D, device=device), 'title': 'Cauchy-like t (df=1)'},
    {'df': 3.0, 'scale_matrix': torch.eye(DIM_2D, device=device), 'title': 'Standard t (df=3)'},
    {'df': 10.0, 'scale_matrix': torch.eye(DIM_2D, device=device), 'title': 'Near-Normal t (df=10)'},
    {'df': 3.0, 'scale_matrix': 0.5**2 * torch.eye(DIM_2D, device=device), 'title': 'Scaled t (df=3, scale=0.5)'}
]

# 2D visualizations
fig, axes = plt.subplots(2, 4, figsize=(20, 10))
fig.suptitle('Student\'s t-Distribution Initializations', fontsize=16)

for i, config in enumerate(t_configs):
    # Create representation layer
    layer = RepresentationLayer(
        dim=DIM_2D, 
        n_samples=N_SAMPLES, 
        dist='student_t',
        dist_params={'df': config['df'], 'scale_matrix': config['scale_matrix']},
        device=device
    )
    
    # Print statistics
    print_statistics(layer.z, config['title'])
    
    # Scatter plot with consistent limits
    plot_2d_scatter(layer.z, config['title'], axes[0, i])
    axes[0, i].set_xlim(-4, 4)
    axes[0, i].set_ylim(-4, 4)
    
    # Histogram with consistent limits and bin width
    plot_histogram(layer.z, config['title'], axes[1, i], xlim=(-6, 6), bin_width=0.5)

plt.tight_layout()
plt.show()

# 3D visualization for standard t-distribution
layer_3d = RepresentationLayer(dim=DIM_3D, n_samples=N_SAMPLES, dist='student_t', device=device)
fig, ax = plot_3d_scatter(layer_3d.z, '3D Student\'s t-Distribution (df=3)')
ax.set_xlim(-4, 4)
ax.set_ylim(-4, 4)
ax.set_zlim(-4, 4)
plt.show()

## Hyperbolic Distribution

Flexible heavy-tailed, potentially asymmetric distribution from the generalized hyperbolic family (normal-variance mixture): $\mathbf{X}=\boldsymbol\mu+\tfrac{\beta\delta^2}{\gamma}W+\delta\sqrt{W}\,\mathbf Z$, where $W\sim\mathrm{GIG}$ (approximated here by a Gamma distribution) and $\mathbf Z\sim\mathcal N(\mathbf 0,\mathbf I_k)$.

Parameters: $\alpha>|\beta|$ (tail heaviness), $\beta$ (asymmetry, $0$ = symmetric), $\delta>0$ (scale). As $\alpha\to\infty,\ \beta=0$, approaches normal. Scalar `mu`/`alpha`/`beta`/`delta` broadcast uniformly.

In [ ]:
print("=== HYPERBOLIC DISTRIBUTION DEMONSTRATION ===")

# Use CPU device for consistency
device = torch.device('cpu')

# Create hyperbolic distributions with different parameters
hyperbolic_configs = [
    {'alpha': 1.0, 'beta': 0.0, 'delta': 1.0, 'title': '(α=1.0, β=0, δ=1)'},
    {'alpha': 3.0, 'beta': 0.0, 'delta': 1.0, 'title': '(α=3.0, β=0, δ=1)'},
    {'alpha': 1.0, 'beta': 0.0, 'delta': 2.0, 'title': '(α=1.0, β=0, δ=2.0)'},
    {'alpha': 2.0, 'beta': 0.0, 'delta': 0.5, 'title': '(α=2.0, β=0, δ=0.5)'}
]

# 2D visualizations
fig, axes = plt.subplots(2, 4, figsize=(20, 10))
fig.suptitle('Hyperbolic Distribution Initializations', fontsize=16)

for i, config in enumerate(hyperbolic_configs):
    # Create representation layer
    layer = RepresentationLayer(
        dim=DIM_2D, 
        n_samples=N_SAMPLES, 
        dist='hyperbolic',
        dist_params={
            'alpha': config['alpha'], 
            'beta': config['beta'], 
            'delta': config['delta']
        },
        device=device
    )
    
    # Print statistics
    print_statistics(layer.z, config['title'])
    
    # Scatter plot with consistent limits
    plot_2d_scatter(layer.z, config['title'], axes[0, i])
    axes[0, i].set_xlim(-6, 6)
    axes[0, i].set_ylim(-6, 6)
    
    # Histogram with consistent limits and bin width
    plot_histogram(layer.z, config['title'], axes[1, i], xlim=(-8, 8), bin_width=0.5)

plt.tight_layout()
plt.show()

# 3D visualization for symmetric hyperbolic
layer_3d = RepresentationLayer(
    dim=DIM_3D, 
    n_samples=N_SAMPLES, 
    dist='hyperbolic',
    dist_params={'alpha': 2.0, 'beta': 0.0, 'delta': 1.0},
    device=device
)
fig, ax = plot_3d_scatter(layer_3d.z, '3D Symmetric Hyperbolic Distribution (α=2, β=0)')
ax.set_xlim(-6, 6)
ax.set_ylim(-6, 6)
ax.set_zlim(-6, 6)
plt.show()

## Logistic Distribution

Independent per-component logistic, heavier-tailed than normal with an S-shaped CDF: $F(x_i)=\mathrm{sigmoid}\big(\tfrac{x_i-\mu_i}{s_i}\big)$, $\mathrm{Var}[X_i]=\pi^2s_i^2/3$. Sampled via inverse-CDF: $X_i=\mu_i+s_i\log\frac{U_i}{1-U_i}$, $U_i\sim\mathrm{Unif}(0,1)$. Scalar `loc`/`scale` broadcast to $\mu\mathbf 1$ / $s\mathbf 1$.

In [ ]:
print("=== LOGISTIC DISTRIBUTION DEMONSTRATION ===")

# Use CPU device for consistency
device = torch.device('cpu')

# Create logistic distributions with different parameters
logistic_configs = [
    {'loc': 0.0, 'scale': 1.0, 'title': 'Standard Logistic (μ=0, s=1)'},
    {'loc': 0.0, 'scale': 0.5, 'title': 'Narrow Logistic (μ=0, s=0.5)'},
    {'loc': 1.0, 'scale': 1.0, 'title': 'Shifted Logistic (μ=1, s=1)'},
    {'loc': 0.0, 'scale': 2.0, 'title': 'Wide Logistic (μ=0, s=2)'}
]

# 2D visualizations
fig, axes = plt.subplots(2, 4, figsize=(20, 10))
fig.suptitle('Logistic Distribution Initializations', fontsize=16)

for i, config in enumerate(logistic_configs):
    # Create representation layer
    layer = RepresentationLayer(
        dim=DIM_2D, 
        n_samples=N_SAMPLES, 
        dist='logistic',
        dist_params={'loc': config['loc'], 'scale': config['scale']},
        device=device
    )
    
    # Print statistics
    print_statistics(layer.z, config['title'])
    
    # Scatter plot with consistent limits
    plot_2d_scatter(layer.z, config['title'], axes[0, i])
    axes[0, i].set_xlim(-6, 6)
    axes[0, i].set_ylim(-6, 6)
    
    # Histogram with consistent limits and bin width
    plot_histogram(layer.z, config['title'], axes[1, i], xlim=(-8, 8), bin_width=0.5)

plt.tight_layout()
plt.show()

# 3D visualization for standard logistic
layer_3d = RepresentationLayer(dim=DIM_3D, n_samples=N_SAMPLES, dist='logistic', device=device)
fig, ax = plot_3d_scatter(layer_3d.z, '3D Standard Logistic Distribution')
ax.set_xlim(-6, 6)
ax.set_ylim(-6, 6)
ax.set_zlim(-6, 6)
plt.show()

# Theoretical vs Empirical Properties Comparison
print(f"\nTheoretical vs Empirical Properties:")
print("="*50)

layer_standard = RepresentationLayer(dim=1, n_samples=10000, dist='logistic', device=device)
data_1d = layer_standard.z.detach().cpu().numpy().flatten()

# Theoretical values for standard logistic (μ=0, s=1)
theoretical_mean = 0.0
theoretical_variance = (np.pi**2) / 3  # π²/3 ≈ 3.29
theoretical_std = np.sqrt(theoretical_variance)

print(f"Mean:      Theoretical = {theoretical_mean:.4f}, Empirical = {data_1d.mean():.4f}")
print(f"Variance:  Theoretical = {theoretical_variance:.4f}, Empirical = {data_1d.var():.4f}")
print(f"Std Dev:   Theoretical = {theoretical_std:.4f}, Empirical = {data_1d.std():.4f}")
print(f"Skewness:  Theoretical = 0.0000, Empirical = {stats.skew(data_1d):.4f}")
print(f"Kurtosis:  Theoretical = 1.2000, Empirical = {stats.kurtosis(data_1d):.4f}")  # Excess kurtosis = 6/5 = 1.2

# Compare with normal distribution of same variance
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
fig.suptitle('Logistic vs Normal Distribution (same variance)', fontsize=14)

# Generate normal with same variance as logistic
normal_std = theoretical_std
normal_samples = torch.randn(10000, device=device) * normal_std
normal_data = normal_samples.detach().cpu().numpy()

# Plot both distributions
plot_histogram(torch.tensor(data_1d).unsqueeze(0), 'Logistic (π²/3 variance)', axes[0], xlim=(-8, 8), bin_width=0.5)
plot_histogram(normal_samples.unsqueeze(0), f'Normal (σ={normal_std:.3f})', axes[1], xlim=(-8, 8), bin_width=0.5)

plt.tight_layout()
plt.show()

print(f"\nTail Comparison:")
print(f"Logistic P(|X| > 3): {2 * (1 - 1/(1 + np.exp(-3))):.4f}")
print(f"Normal P(|X| > 3):   {2 * (1 - stats.norm.cdf(3/normal_std)):.4f}")
print(f"→ Logistic has heavier tails than normal")

## Distribution Comparison Analysis

Compare all distributions side-by-side with standardized parameters.

In [ ]:
# Use CPU device for consistency
device = torch.device('cpu')

# Create all distributions with standardized parameters in the new order:
# uniform, uniform ball, uniform sphere, normal, student, laplace, hyperbolic, logistic, zeros
distributions = {
    'Uniform': RepresentationLayer(dim=DIM_2D, n_samples=N_SAMPLES, dist='uniform', device=device),
    'Uniform Ball': RepresentationLayer(dim=DIM_2D, n_samples=N_SAMPLES, dist='uniform_ball', device=device),
    'Uniform Sphere': RepresentationLayer(dim=DIM_2D, n_samples=N_SAMPLES, dist='uniform_sphere', device=device),
    'Normal': RepresentationLayer(dim=DIM_2D, n_samples=N_SAMPLES, dist='normal', device=device),
    'Student t': RepresentationLayer(dim=DIM_2D, n_samples=N_SAMPLES, dist='student_t', device=device),
    'Laplace': RepresentationLayer(dim=DIM_2D, n_samples=N_SAMPLES, dist='laplace', device=device),
    'Hyperbolic': RepresentationLayer(dim=DIM_2D, n_samples=N_SAMPLES, dist='hyperbolic', device=device),
    'Logistic': RepresentationLayer(dim=DIM_2D, n_samples=N_SAMPLES, dist='logistic', device=device),
    'Zeros': RepresentationLayer(dim=DIM_2D, n_samples=N_SAMPLES, dist='zeros', device=device)
}

# 2D Scatter plot comparison
fig, axes = plt.subplots(3, 3, figsize=(18, 18))
fig.suptitle('Distribution Comparison - 2D Scatter Plots', fontsize=16)
axes = axes.flatten()

for i, (name, layer) in enumerate(distributions.items()):
    if i < len(axes):
        plot_2d_scatter(layer.z, name, axes[i])
        # Set consistent axis limits (special cases for some distributions)
        if name == 'Zeros':
            axes[i].set_xlim(-0.1, 0.1)
            axes[i].set_ylim(-0.1, 0.1)
        elif name in ['Uniform Sphere', 'Uniform Ball']:
            axes[i].set_xlim(-1.5, 1.5)
            axes[i].set_ylim(-1.5, 1.5)
            axes[i].set_aspect('equal')
            # Add circle for sphere/ball visualization
            circle = plt.Circle((0, 0), 1.0, fill=False, color='red', linestyle='--', linewidth=1)
            axes[i].add_patch(circle)
        else:
            axes[i].set_xlim(-4, 4)
            axes[i].set_ylim(-4, 4)

plt.tight_layout()
plt.show()

# Histogram comparison
fig, axes = plt.subplots(3, 3, figsize=(18, 18))
fig.suptitle('Distribution Comparison - Histograms', fontsize=16)
axes = axes.flatten()

for i, (name, layer) in enumerate(distributions.items()):
    if i < len(axes):
        if name == 'Zeros':  # Zeros needs special handling
            plot_histogram(layer.z, name, axes[i], xlim=(-0.1, 0.1), bin_width=0.01)
        elif name in ['Hyperbolic', 'Logistic']:  # Heavy-tailed distributions
            plot_histogram(layer.z, name, axes[i], xlim=(-8, 8), bin_width=0.5)
        else:  # Standard range for most distributions
            plot_histogram(layer.z, name, axes[i], xlim=(-5, 5), bin_width=0.25)

plt.tight_layout()
plt.show()

# Statistical summary table
print("\n" + "="*90)
print("STATISTICAL SUMMARY TABLE")
print("="*90)
print(f"{'Distribution':<15} {'Mean':<8} {'Std':<8} {'Min':<8} {'Max':<8} {'Skew':<8} {'Kurt':<8} {'Range':<8}")
print("-"*90)

for name, layer in distributions.items():
    data_flat = layer.z.detach().cpu().numpy().flatten()
    data_range = data_flat.max() - data_flat.min()
    print(f"{name:<15} {data_flat.mean():<8.3f} {data_flat.std():<8.3f} {data_flat.min():<8.3f} {data_flat.max():<8.3f} {stats.skew(data_flat):<8.3f} {stats.kurtosis(data_flat):<8.3f} {data_range:<8.3f}")

print("-"*90)

# Distribution family groupings
print(f"\nDistribution Family Characteristics:")
print("="*50)
print("  UNIFORM FAMILY:")
print("  • Uniform: Rectangular, bounded support, constant density")
print("  • Uniform Ball: Spherical volume, radial concentration in high dims")  
print("  • Uniform Sphere: Surface only, all points equidistant from center")

print("\n  GAUSSIAN FAMILY:")
print("  • Normal: Light tails, maximum entropy for given variance")
print("  • Student t: Heavy tails, robust to outliers")
print("  • Laplace: Exponential tails, sparsity-promoting")

print("\n  Other:")
print("  • Hyperbolic: Heavy tails, can be asymmetrical")
print("  • Logistic: Heavy tails, S-shaped CDF, sigmoid-related")
print("  • Zeros: Deterministic initialization at origin")


## PCA Initialization Demo

Demonstrate PCA-based initialization using real image data. PCA initialization projects high-dimensional data onto its principal components, providing a data-driven starting point for representation learning.

In [ ]:
# PCA Initialization Demonstration
import torchvision
import torchvision.transforms as transforms

print("Loading FashionMNIST dataset for PCA demonstration...")
# Load FashionMNIST using torchvision (will use existing data if already downloaded)
transform = transforms.Compose([transforms.ToTensor()])
fmnist_data = torchvision.datasets.FashionMNIST(
    root='../data',  # Uses parent directory
    train=True, 
    download=True, 
    transform=transform
)

# Create a dataloader for sampling
from torch.utils.data import DataLoader, Subset
n_samples_pca = 1000
indices = torch.randperm(len(fmnist_data))[:n_samples_pca]
subset = Subset(fmnist_data, indices)
pca_loader = DataLoader(subset, batch_size=n_samples_pca, shuffle=False)

# Extract data from loader
subset_data, subset_labels = next(iter(pca_loader))
data_flat = subset_data.reshape(n_samples_pca, -1)  # Shape: (1000, 784)

print(f"Data shape: {data_flat.shape}")
print(f"Labels shape: {subset_labels.shape}")

# Analyze variance for different numbers of components (2 to 10)
print("\nAnalyzing explained variance for 2-10 components...")
component_range = range(2, 11)
variance_results = []

for n_comp in component_range:
    rep_temp = RepresentationLayer(
        dim=n_comp,
        n_samples=n_samples_pca,
        dist='pca',
        dist_params={'data': data_flat, 'whiten': False},
        device=torch.device('cpu')
    )
    total_var = sum(rep_temp._options.get('explained_variance_ratio', [])) * 100
    variance_results.append(total_var)
    print(f"  {n_comp} components: {total_var:.2f}% variance explained")

# Create PCA with 2 components for visualization
print("\nCreating 2-component PCA for visualization...")
rep_pca = RepresentationLayer(
    dim=2,
    n_samples=n_samples_pca,
    dist='pca',
    dist_params={'data': data_flat, 'whiten': False},
    device=torch.device('cpu')
)

print(f"✓ PCA initialization complete")
print(f"  Representation shape: {rep_pca.z.shape}")
print(f"  Total variance captured (2 components): {sum(rep_pca._options['explained_variance_ratio']) * 100:.2f}%")

# Create visualization
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Plot 1: First 2 PCA components colored by class
pca_np = rep_pca.z.detach().cpu().numpy()
scatter = axes[0].scatter(
    pca_np[:, 0], pca_np[:, 1], 
    c=subset_labels.numpy(), 
    cmap='tab10', 
    alpha=0.6, 
    s=20,
    edgecolors='none'
)
axes[0].set_xlabel(f'PC 1 ({rep_pca._options["explained_variance_ratio"][0]*100:.1f}%)', fontsize=11)
axes[0].set_ylabel(f'PC 2 ({rep_pca._options["explained_variance_ratio"][1]*100:.1f}%)', fontsize=11)
axes[0].set_title('PCA Projection (2 Components)', fontsize=13, fontweight='bold')
axes[0].grid(True, alpha=0.3)
cbar = plt.colorbar(scatter, ax=axes[0])
cbar.set_label('Fashion Class', fontsize=10)

# Plot 2: Variance explained by 2-10 components
comp_numbers = list(component_range)
bars = axes[1].bar(comp_numbers, variance_results, alpha=0.7, color='steelblue', edgecolor='black', linewidth=1.5)

# Add value labels on bars
for i, (comp, var) in enumerate(zip(comp_numbers, variance_results)):
    axes[1].text(comp, var + 1, f'{var:.1f}%', ha='center', va='bottom', fontsize=9, fontweight='bold')

# Add reference lines
axes[1].axhline(y=50, color='orange', linestyle='--', linewidth=2, label='50% variance', alpha=0.7)
axes[1].axhline(y=70, color='green', linestyle='--', linewidth=2, label='70% variance', alpha=0.7)
axes[1].axhline(y=90, color='red', linestyle='--', linewidth=2, label='90% variance', alpha=0.7)

axes[1].set_xlabel('Number of Components', fontsize=11)
axes[1].set_ylabel('Cumulative Explained Variance (%)', fontsize=11)
axes[1].set_title('Explained Variance by Component Count', fontsize=13, fontweight='bold')
axes[1].set_xticks(comp_numbers)
axes[1].set_ylim([0, 100])
axes[1].legend(loc='lower right', fontsize=9)
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

In [ ]:
# Train/Test PCA Comparison for Multiple Dimensions
print("="*80)
print("PCA TRAIN/TEST COMPARISON - Multiple Dimensions")
print("="*80)

# Load FashionMNIST train and test datasets
transform = transforms.Compose([transforms.ToTensor()])

train_dataset = torchvision.datasets.FashionMNIST(
    root='../data',
    train=True,
    download=True,
    transform=transform
)

test_dataset = torchvision.datasets.FashionMNIST(
    root='../data',
    train=False,
    download=True,
    transform=transform
)

# Sample from train and test
n_samples_comparison = 1000
train_indices = torch.randperm(len(train_dataset))[:n_samples_comparison]
test_indices = torch.randperm(len(test_dataset))[:n_samples_comparison]

train_subset = Subset(train_dataset, train_indices)
test_subset = Subset(test_dataset, test_indices)

train_loader_pca = DataLoader(train_subset, batch_size=n_samples_comparison, shuffle=False)
test_loader_pca = DataLoader(test_subset, batch_size=n_samples_comparison, shuffle=False)

# Extract data
train_data, train_labels = next(iter(train_loader_pca))
test_data, test_labels = next(iter(test_loader_pca))

train_data_flat = train_data.reshape(n_samples_comparison, -1)
test_data_flat = test_data.reshape(n_samples_comparison, -1)

print(f"\nTrain data shape: {train_data_flat.shape}")
print(f"Test data shape: {test_data_flat.shape}")

# Dimensions to compare
dimensions = [2, 4, 8]

# Create figure with subplots
fig = plt.figure(figsize=(20, 12))
gs = fig.add_gridspec(3, 3, hspace=0.3, wspace=0.3)

for row, n_dim in enumerate(dimensions):
    print(f"\n{'='*60}")
    print(f"Creating PCA with {n_dim} components...")
    
    # Create separate PCA for train and test
    rep_train = RepresentationLayer(
        dim=n_dim,
        n_samples=n_samples_comparison,
        dist='pca',
        dist_params={'data': train_data_flat, 'whiten': False},
        device=torch.device('cpu')
    )
    
    rep_test = RepresentationLayer(
        dim=n_dim,
        n_samples=n_samples_comparison,
        dist='pca',
        dist_params={'data': test_data_flat, 'whiten': False},
        device=torch.device('cpu')
    )
    
    train_var = sum(rep_train._options['explained_variance_ratio']) * 100
    test_var = sum(rep_test._options['explained_variance_ratio']) * 100
    
    print(f"  Train variance explained: {train_var:.2f}%")
    print(f"  Test variance explained: {test_var:.2f}%")
    
    # Get numpy arrays for plotting (use first 2 dimensions for visualization)
    train_pca_np = rep_train.z.detach().cpu().numpy()
    test_pca_np = rep_test.z.detach().cpu().numpy()
    train_labels_np = train_labels.numpy()
    test_labels_np = test_labels.numpy()
    
    # Plot 1: Train dataset (first 2 PCs)
    ax1 = fig.add_subplot(gs[row, 0])
    scatter1 = ax1.scatter(
        train_pca_np[:, 0], train_pca_np[:, 1],
        c=train_labels_np,
        cmap='tab10',
        alpha=0.6,
        s=15,
        edgecolors='none'
    )
    ax1.set_xlabel(f'PC 1 ({rep_train._options["explained_variance_ratio"][0]*100:.1f}%)', fontsize=10)
    ax1.set_ylabel(f'PC 2 ({rep_train._options["explained_variance_ratio"][1]*100:.1f}%)', fontsize=10)
    ax1.set_title(f'Train PCA ({n_dim}D) - Total: {train_var:.1f}%', fontsize=11, fontweight='bold')
    ax1.grid(True, alpha=0.3)
    
    # Plot 2: Test dataset (first 2 PCs)
    ax2 = fig.add_subplot(gs[row, 1])
    scatter2 = ax2.scatter(
        test_pca_np[:, 0], test_pca_np[:, 1],
        c=test_labels_np,
        cmap='tab10',
        alpha=0.6,
        s=15,
        edgecolors='none'
    )
    ax2.set_xlabel(f'PC 1 ({rep_test._options["explained_variance_ratio"][0]*100:.1f}%)', fontsize=10)
    ax2.set_ylabel(f'PC 2 ({rep_test._options["explained_variance_ratio"][1]*100:.1f}%)', fontsize=10)
    ax2.set_title(f'Test PCA ({n_dim}D) - Total: {test_var:.1f}%', fontsize=11, fontweight='bold')
    ax2.grid(True, alpha=0.3)
    
    # Plot 3: Variance comparison bar chart
    ax3 = fig.add_subplot(gs[row, 2])
    
    # Get per-component variance
    train_var_per_comp = [v * 100 for v in rep_train._options['explained_variance_ratio']]
    test_var_per_comp = [v * 100 for v in rep_test._options['explained_variance_ratio']]
    
    x = np.arange(n_dim)
    width = 0.35
    
    bars1 = ax3.bar(x - width/2, train_var_per_comp, width, label='Train', alpha=0.8, color='steelblue', edgecolor='black')
    bars2 = ax3.bar(x + width/2, test_var_per_comp, width, label='Test', alpha=0.8, color='coral', edgecolor='black')
    
    ax3.set_xlabel('Component', fontsize=10)
    ax3.set_ylabel('Variance Explained (%)', fontsize=10)
    ax3.set_title(f'Variance per Component ({n_dim}D)', fontsize=11, fontweight='bold')
    ax3.set_xticks(x)
    ax3.set_xticklabels([f'PC{i+1}' for i in range(n_dim)])
    ax3.legend(fontsize=9)
    ax3.grid(True, alpha=0.3, axis='y')
    
    # Add cumulative variance text
    ax3.text(0.98, 0.98, 
             f'Cumulative:\nTrain: {train_var:.1f}%\nTest: {test_var:.1f}%',
             transform=ax3.transAxes,
             fontsize=9,
             verticalalignment='top',
             horizontalalignment='right',
             bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

# Add colorbar
cbar = fig.colorbar(scatter2, ax=fig.get_axes(), orientation='vertical', fraction=0.02, pad=0.02)
cbar.set_label('Fashion Class', fontsize=10)

# Add overall title
fig.suptitle('Train vs Test PCA Comparison - FashionMNIST', fontsize=16, fontweight='bold', y=0.995)

plt.show()

# Print summary table
print(f"\n{'='*80}")
print("SUMMARY: Variance Explained by Dimension")
print(f"{'='*80}")
print(f"{'Dimensions':<12} {'Train Variance':<18} {'Test Variance':<18} {'Difference':<12}")
print(f"{'-'*80}")
for n_dim in dimensions:
    # Recreate for summary
    rep_train = RepresentationLayer(
        dim=n_dim, n_samples=n_samples_comparison, dist='pca',
        dist_params={'data': train_data_flat, 'whiten': False},
        device=torch.device('cpu')
    )
    rep_test = RepresentationLayer(
        dim=n_dim, n_samples=n_samples_comparison, dist='pca',
        dist_params={'data': test_data_flat, 'whiten': False},
        device=torch.device('cpu')
    )
    train_var = sum(rep_train._options['explained_variance_ratio']) * 100
    test_var = sum(rep_test._options['explained_variance_ratio']) * 100
    diff = abs(train_var - test_var)
    print(f"{n_dim:2d}D          {train_var:6.2f}%            {test_var:6.2f}%            {diff:5.2f}%")

print(f"{'-'*80}")
print("\nKey Observations:")
print("  • PCA is fitted separately on train and test data")
print("  • Both datasets show similar variance patterns")
print("  • Small differences indicate consistent data structure")
print("  • Higher dimensions capture more total variance")
print("  • First 2 PCs are plotted for visualization across all dimensions")

## PCA Sign Ambiguity

If $\mathbf v$ is an eigenvector of the covariance matrix with eigenvalue $\lambda$, so is $-\mathbf v$ ($\Sigma(-\mathbf v)=\lambda(-\mathbf v)$) -- different fits (train vs. test, or different runs) can pick either sign, which is why the projections above can appear mirrored. Variance explained, pairwise distances, and cluster structure are all unaffected by the flip.